In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Task 1: EDA and Preprocessing - CFPB Complaint Data\n",
    "\n",
    "**Objective:** Load the full CFPB dataset, explore its structure, filter for relevant products, clean the text narratives, and save the processed data.\n",
    "\n",
    "**Products of interest:** Credit Card, Personal Loan, Savings Account, Money Transfer\n",
    "\n",
    "**Deliverables:**\n",
    "- This notebook\n",
    "- A 2-3 paragraph summary of key findings (included below)\n",
    "- `data/processed/filtered_complaints.csv`"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "import re\n",
    "from pathlib import Path\n",
    "\n",
    "# Set display options\n",
    "pd.set_option('display.max_colwidth', 200)\n",
    "pd.set_option('display.max_rows', 100)\n",
    "\n",
    "# Project paths\n",
    "RAW_DATA_PATH = Path(\"data/raw/complaints.csv\")\n",
    "PROCESSED_DATA_PATH = Path(\"data/processed/filtered_complaints.csv\")\n",
    "\n",
    "print(f\"Raw data exists? {RAW_DATA_PATH.exists()}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Load the dataset"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load the CSV (adjust column names if needed)\n",
    "df = pd.read_csv(RAW_DATA_PATH, low_memory=False)\n",
    "print(f\"Loaded {len(df)} records.\")\n",
    "df.head()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Initial EDA"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Check columns and data types\n",
    "df.info()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Distribution of complaints per product\n",
    "product_col = 'product' if 'product' in df.columns else 'product_category'\n",
    "\n",
    "product_counts = df[product_col].value_counts()\n",
    "print(\"Product distribution:\")\n",
    "print(product_counts)\n",
    "\n",
    "# Plot\n",
    "plt.figure(figsize=(10,6))\n",
    "product_counts.plot(kind='bar')\n",
    "plt.title('Complaints by Product Category')\n",
    "plt.xlabel('Product')\n",
    "plt.ylabel('Number of Complaints')\n",
    "plt.xticks(rotation=45)\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Consumer narrative presence\n",
    "narrative_col = 'consumer_complaint_narrative' if 'consumer_complaint_narrative' in df.columns else 'narrative'\n",
    "\n",
    "if narrative_col:\n",
    "    missing_narratives = df[narrative_col].isna().sum()\n",
    "    print(f\"Records without narrative: {missing_narratives} ({missing_narratives/len(df)*100:.2f}%)\")\n",
    "    \n",
    "    # Narrative length distribution\n",
    "    df['narrative_word_count'] = df[narrative_col].astype(str).apply(lambda x: len(x.split()))\n",
    "    print(f\"\\nNarrative length stats (word count):\")\n",
    "    print(df['narrative_word_count'].describe())\n",
    "\n",
    "    # Plot histogram\n",
    "    plt.figure(figsize=(10,4))\n",
    "    df['narrative_word_count'].hist(bins=50, edgecolor='black')\n",
    "    plt.title('Distribution of Narrative Length (word count)')\n",
    "    plt.xlabel('Word Count')\n",
    "    plt.ylabel('Frequency')\n",
    "    plt.xlim(0, 500)\n",
    "    plt.show()\n",
    "else:\n",
    "    print(\"Narrative column not found. Check column names.\")\n",
    "    print(\"Available columns:\", df.columns.tolist())"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Filter Dataset"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Define the four target products\n",
    "target_products = ['Credit Card', 'Personal Loan', 'Savings Account', 'Money Transfer']\n",
    "\n",
    "# Filter\n",
    "df_filtered = df[df[product_col].isin(target_products)].copy()\n",
    "print(f\"Records after filtering by product: {len(df_filtered)}\")\n",
    "\n",
    "# Drop rows without narrative\n",
    "if narrative_col:\n",
    "    before = len(df_filtered)\n",
    "    df_filtered = df_filtered.dropna(subset=[narrative_col])\n",
    "    print(f\"Dropped {before - len(df_filtered)} rows with empty narratives.\")\n",
    "    print(f\"Final filtered dataset size: {len(df_filtered)}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Check distribution in filtered set\n",
    "df_filtered[product_col].value_counts()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Clean Text Narratives"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def clean_text(text):\n",
    "    if not isinstance(text, str):\n",
    "        return \"\"\n",
    "    \n",
    "    text = text.lower()\n",
    "    text = re.sub(r'[^a-zA-Z0-9\\s\\.\\,\\?\\!]', '', text)\n",
    "    \n",
    "    boilerplates = [\n",
    "        \"i am writing to file a complaint\",\n",
    "        \"i am writing to complain about\",\n",
    "        \"i am writing to notify you\",\n",
    "        \"this is a complaint regarding\",\n",
    "        \"i am submitting this complaint because\"\n",
    "    ]\n",
    "    for phrase in boilerplates:\n",
    "        text = text.replace(phrase, \"\")\n",
    "    \n",
    "    text = re.sub(r'\\s+', ' ', text).strip()\n",
    "    return text\n",
    "\n",
    "# Apply cleaning\n",
    "if narrative_col:\n",
    "    df_filtered['cleaned_narrative'] = df_filtered[narrative_col].apply(clean_text)\n",
    "    \n",
    "    # Check a sample\n",
    "    print(\"Sample original vs cleaned:\")\n",
    "    for i in range(3):\n",
    "        print(f\"\\nOriginal: {df_filtered[narrative_col].iloc[i][:200]}...\")\n",
    "        print(f\"Cleaned:  {df_filtered['cleaned_narrative'].iloc[i][:200]}...\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Save the Filtered and Cleaned Dataset"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Ensure the processed directory exists\n",
    "PROCESSED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)\n",
    "\n",
    "# Save\n",
    "df_filtered.to_csv(PROCESSED_DATA_PATH, index=False)\n",
    "print(f\"Saved filtered dataset to {PROCESSED_DATA_PATH}\")\n",
    "print(f\"Shape: {df_filtered.shape}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Summary of Key Findings\n",
    "\n",
    "*(Write your 2-3 paragraph summary here after running the notebook.)*\n",
    "\n",
    "**Example:**\n",
    "- The original dataset contains X million complaints, with the majority falling under Credit Card and Personal Loan products.\n",
    "- Approximately Y% of records contain a consumer narrative; the rest are missing, which limits our ability to extract insights.\n",
    "- Narrative lengths vary widely: the average is ~Z words, with some extremely long entries (>1000 words) that will require chunking.\n",
    "- After filtering for the four target products and removing empty narratives, we retain N records.\n",
    "- Common boilerplate phrases are present and have been removed to improve the signal-to-noise ratio for embedding and retrieval."
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.10.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}